# Simplified Context Features Extraction

This notebook extracts context saturation metrics for two scenarios:
1. **With context**: `context_text = "\n\n".join(backgrounds)`
2. **Without context**: `context_text = "No"`

## Output Structure

Each row contains metrics at the top level (matching original pattern):
```json
{
  "id": "sample_id",
  "question": "...",
  "gold_answer": "...",
  "task_type": "open_qa",
  "mid_context_metrics": {...},      // Mid-layer with full context
  "last_context_metrics": {...},     // Last-layer with full context
  "mid_no_context_metrics": {...},   // Mid-layer with "No"
  "last_no_context_metrics": {...}   // Last-layer with "No"
}
```

Each metrics dict contains: `l2_mean`, `l2_std`, `hoyer_mean`, `hoyer_std`, `spec_entropy_mean`, `spec_entropy_std`, `excess_kurtosis_mean`, `excess_kurtosis_std`.


In [5]:
import os

# Config: set paths and optional CUDA device (None = use default)
BASE_PATH = "/app/xlong/scripts/data_preprocessing/runs/squad"                         # root dir for inputs/outputs
INPUT_SAMPLES_PATH = f"{BASE_PATH}/samples.jsonl"   # JSONL: id, question, background, gold_answer
OUTPUT_DIR = f"{BASE_PATH}"                 # directory for context_features.jsonl
XRAG_DIR = "../xRAG"                                # path to xRAG repo (model imports)
CUDA_DEVICE = None                                 # e.g. "0"; None = default
if CUDA_DEVICE is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_DEVICE

In [6]:
import torch

def tensor_to_numpy(tensor: torch.Tensor | None):
    """Detach tensor to CPU numpy, promoting low precision floats if needed."""
    if tensor is None:
        return None
    if not isinstance(tensor, torch.Tensor):
        return tensor
    if tensor.is_floating_point() and tensor.dtype in {torch.bfloat16, torch.float16}:
        tensor = tensor.to(torch.float32)
    return tensor.detach().cpu().numpy()

In [7]:
# Setup paths and device (XRAG_DIR, OUTPUT_DIR from config above)
import sys
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

assert os.path.isdir(XRAG_DIR), f"XRAG_DIR not found: {XRAG_DIR}"
sys.path.insert(0, XRAG_DIR)


Device: cuda


In [8]:
# Load input samples (JSONL with id, question, background, gold_answer)
import json

samples_orig = []
with open(INPUT_SAMPLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        samples_orig.append(json.loads(line))

print(f"Loaded {len(samples_orig)} samples")


Loaded 100 samples


In [9]:
# Load model
from transformers import AutoTokenizer
from src.model import XMistralForCausalLM

model_path = "/app/models/xrag-7b"
print(f"Loading LLM from {model_path}...")

model = XMistralForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto",
    attn_implementation="eager",
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    padding_side="left",
    add_eos_token=False,
    use_fast=False,
)

if tokenizer.pad_token:
    pass
elif tokenizer.unk_token:
    tokenizer.pad_token_id = tokenizer.unk_token_id
elif tokenizer.eos_token:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Model loaded successfully")


Loading LLM from /app/models/xrag-7b...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully


In [10]:
# Saturation metrics computation
import math
from scipy.fft import dct
from scipy.stats import kurtosis

def compute_group_saturation_metrics(hidden_states, token_indices):
    """
    Compute saturation metrics for a group of tokens.
    
    Args:
        hidden_states: [seq_len, hidden_dim] tensor
        token_indices: list/tensor of token positions to analyze
    
    Returns:
        dict with group-level metrics
    """
    if not isinstance(token_indices, torch.Tensor):
        token_indices = torch.tensor(token_indices, dtype=torch.long)
    
    # Extract vectors for the group
    group_vecs = hidden_states[token_indices]  # [n_tokens, hidden_dim]
    
    if group_vecs.numel() == 0:
        return None
    
    # Convert to numpy for scipy functions
    group_np = tensor_to_numpy(group_vecs.detach().cpu())
    
    metrics = {}
    
    # L2 norm
    l2_norms = np.linalg.norm(group_np, axis=1)
    metrics["l2_first"] = float(l2_norms[0])
    metrics["l2_mean"] = float(np.mean(l2_norms))
    metrics["l2_std"] = float(np.std(l2_norms))
    
    # Hoyer's sparsity
    def hoyer_sparsity(vec):
        d = len(vec)
        l1 = np.sum(np.abs(vec))
        l2 = np.linalg.norm(vec)
        if l2 < 1e-9:
            return 0.0
        return (math.sqrt(d) - (l1 / l2)) / (math.sqrt(d) - 1)
    
    hoyers = [hoyer_sparsity(vec) for vec in group_np]
    metrics["hoyer_first"] = float(hoyers[0])
    metrics["hoyer_mean"] = float(np.mean(hoyers))
    metrics["hoyer_std"] = float(np.std(hoyers))
    
    # Spectral entropy
    def spectral_entropy(vec):
        dct_coeffs = dct(vec, norm='ortho')
        power = dct_coeffs ** 2
        power = power / (np.sum(power) + 1e-9)
        entropy = -np.sum(power * np.log(power + 1e-9))
        return entropy
    
    entropies = [spectral_entropy(vec) for vec in group_np]
    metrics["spec_entropy_first"] = float(entropies[0])
    metrics["spec_entropy_mean"] = float(np.mean(entropies))
    metrics["spec_entropy_std"] = float(np.std(entropies))
    
    # Excess kurtosis
    kurtoses = [kurtosis(vec) for vec in group_np]
    metrics["excess_kurtosis_first"] = float(kurtoses[0])
    metrics["excess_kurtosis_mean"] = float(np.mean(kurtoses))
    metrics["excess_kurtosis_std"] = float(np.std(kurtoses))
    
    return metrics

print("Metrics functions defined")


Metrics functions defined


In [11]:
# Prompt template and helper functions


def prompt_context_mistral(context: str, question: str) -> str:
    """Build prompt with context."""
    return f"<s>[INST] Context:\n{context}\n\nQuestion: {question}\n[/INST] The answer is:"


def find_context_token_positions(prompt: str, context_text: str, tokenizer):
    """
    Find token positions corresponding to the context text in the prompt.
    
    Returns:
        list of token indices
    """
    # Tokenize full prompt
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = inputs["input_ids"][0]
    
    # Find where context starts and ends in character positions
    context_header = "<s>[INST] Context:\n"
    context_start_char = prompt.find(context_header)
    if context_start_char == -1:
        raise ValueError("Could not find 'Context:' in prompt")
    
    context_start_char += len(context_header)
    
    # Find where "Question:" starts (marks end of context)
    question_marker = "\n\nQuestion:"
    question_start_char = prompt.find(question_marker, context_start_char)
    if question_start_char == -1:
        question_marker = "Question:"
        question_start_char = prompt.find(question_marker, context_start_char)
    
    if question_start_char == -1:
        raise ValueError("Could not find 'Question:' in prompt")
    
    # Tokenize segments to find token boundaries
    prompt_before_context = prompt[:context_start_char]
    tokens_before_context = tokenizer(prompt_before_context, add_special_tokens=False)["input_ids"]
    context_start_idx = len(tokens_before_context)
    
    prompt_before_question = prompt[:question_start_char]
    tokens_before_question = tokenizer(prompt_before_question, add_special_tokens=False)["input_ids"]
    context_end_idx = len(tokens_before_question)
    
    # Return indices of context tokens
    context_indices = list(range(context_start_idx, context_end_idx))
    
    return context_indices

print("Helper functions defined")


Helper functions defined


In [12]:
# Extract features for one sample

def extract_context_features(sample, model, tokenizer, mid_layer_index=16, debug=False):
    """
    Extract mid_group_metrics and last_group_metrics for both context scenarios.
    
    Args:
        sample: Sample dict with question and background
        model: XMistral model
        tokenizer: Tokenizer
        mid_layer_index: Which layer to use for mid-layer features (default: 16)
        debug: If True, print debug information about context and tokens
    
    Returns:
        dict with metrics at top level matching pattern:
        {
            "mid_context_metrics": {...},
            "last_context_metrics": {...},
            "mid_no_context_metrics": {...},
            "last_no_context_metrics": {...}
        }
    """
    question = sample["question"].strip()
    backgrounds = sample.get("background", [])
    
    results = {}
    
    # ========================================================================
    # Scenario 1: WITH context (backgrounds joined)
    # ========================================================================
    context_text_with = "\n\n".join(backgrounds)
    prompt_with = prompt_context_mistral(context_text_with, question)
    
    if debug:
        print("\n" + "="*80)
        print("SCENARIO 1: WITH CONTEXT")
        print("="*80)
        print(f"\n[Context Text] ({len(context_text_with)} chars):")
        print(context_text_with[:200] + "..." if len(context_text_with) > 200 else context_text_with)
        print(f"\n[Full Prompt] ({len(prompt_with)} chars):")
        print(prompt_with[:300] + "..." if len(prompt_with) > 300 else prompt_with)
    
    inputs_with = tokenizer(prompt_with, return_tensors="pt", add_special_tokens=False)
    inputs_with = {k: v.to(device) for k, v in inputs_with.items()}
    
    # Find context token positions
    context_indices_with = find_context_token_positions(prompt_with, context_text_with, tokenizer)
    
    if debug:
        print(f"\n[Token Positions]")
        print(f"  Total tokens in prompt: {inputs_with['input_ids'].shape[1]}")
        print(f"  Context token indices: {context_indices_with}")
        print(f"  Number of context tokens: {len(context_indices_with)}")
        
        # Show actual context tokens
        if len(context_indices_with) > 0:
            all_tokens = tokenizer.convert_ids_to_tokens(inputs_with['input_ids'][0].cpu().tolist())
            context_tokens = [all_tokens[i] for i in context_indices_with[:10]]  # Show first 10
            print(f"  First context tokens: {context_tokens}")
            decoded_context = tokenizer.decode(inputs_with['input_ids'][0][context_indices_with].cpu())
            print(f"  Decoded context: {decoded_context[:200]}..." if len(decoded_context) > 200 else f"  Decoded context: {decoded_context}")
    
    # Forward pass
    with torch.no_grad():
        outputs_with = model(**inputs_with, output_hidden_states=True)
    
    hidden_states_with = outputs_with.hidden_states
    mid_h_with = hidden_states_with[mid_layer_index][0]  # [seq_len, hidden_dim]
    last_h_with = hidden_states_with[-1][0]
    
    # Compute group metrics for WITH context
    results["mid_context_metrics"] = compute_group_saturation_metrics(mid_h_with, context_indices_with)
    results["last_context_metrics"] = compute_group_saturation_metrics(last_h_with, context_indices_with)
    
    if debug:
        print(f"\n[Computed Metrics]")
        print(f"  Mid-layer: {results['mid_context_metrics']}")
        print(f"  Last-layer: {results['last_context_metrics']}")
    
    # ========================================================================
    # Scenario 2: WITHOUT context ("No")
    # ========================================================================
    context_text_without = "No"
    prompt_without = prompt_context_mistral(context_text_without, question)
    
    if debug:
        print("\n" + "="*80)
        print("SCENARIO 2: WITHOUT CONTEXT")
        print("="*80)
        print(f"\n[Context Text]: '{context_text_without}'")
        print(f"\n[Full Prompt] ({len(prompt_without)} chars):")
        print(prompt_without)
    
    inputs_without = tokenizer(prompt_without, return_tensors="pt", add_special_tokens=False)
    inputs_without = {k: v.to(device) for k, v in inputs_without.items()}
    
    # Find context token positions
    context_indices_without = find_context_token_positions(prompt_without, context_text_without, tokenizer)
    
    if debug:
        print(f"\n[Token Positions]")
        print(f"  Total tokens in prompt: {inputs_without['input_ids'].shape[1]}")
        print(f"  Context token indices: {context_indices_without}")
        print(f"  Number of context tokens: {len(context_indices_without)}")
        
        # Show actual context tokens
        if len(context_indices_without) > 0:
            all_tokens = tokenizer.convert_ids_to_tokens(inputs_without['input_ids'][0].cpu().tolist())
            context_tokens = [all_tokens[i] for i in context_indices_without]
            print(f"  Context tokens: {context_tokens}")
            decoded_context = tokenizer.decode(inputs_without['input_ids'][0][context_indices_without].cpu())
            print(f"  Decoded context: '{decoded_context}'")
    
    # Forward pass
    with torch.no_grad():
        outputs_without = model(**inputs_without, output_hidden_states=True)
    
    hidden_states_without = outputs_without.hidden_states
    mid_h_without = hidden_states_without[mid_layer_index][0]
    last_h_without = hidden_states_without[-1][0]
    
    # Compute group metrics for WITHOUT context
    results["mid_no_context_metrics"] = compute_group_saturation_metrics(mid_h_without, context_indices_without)
    results["last_no_context_metrics"] = compute_group_saturation_metrics(last_h_without, context_indices_without)
    
    if debug:
        print(f"\n[Computed Metrics]")
        print(f"  Mid-layer: {results['mid_no_context_metrics']}")
        print(f"  Last-layer: {results['last_no_context_metrics']}")
        print("="*80 + "\n")
    
    return results

print("Extraction function defined")

print("\n🔍 Debug mode available: set debug=True to visualize context and tokens")


Extraction function defined

🔍 Debug mode available: set debug=True to visualize context and tokens


In [13]:
# Test on one sample with debug output
test_sample = samples_orig[0]
print(f"\n{'='*80}")
print(f"TESTING ON SAMPLE: {test_sample['id']}")
print(f"Question: {test_sample['question']}")
print(f"{'='*80}")

# Run with debug=True to see detailed output
test_results = extract_context_features(test_sample, model, tokenizer, debug=True)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print("\nWith context:")
print("  Mid-layer metrics:", test_results["mid_context_metrics"])
print("  Last-layer metrics:", test_results["last_context_metrics"])
print("\nWithout context:")
print("  Mid-layer metrics:", test_results["mid_no_context_metrics"])
print("  Last-layer metrics:", test_results["last_no_context_metrics"])

print("\n💡 Tip: Debug output shows:")
print("   - Context text (with truncation for long texts)")
print("   - Full prompts sent to the model")
print("   - Token positions and indices")
print("   - Decoded context tokens")
print("   - All computed metrics")


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)



TESTING ON SAMPLE: 56ddde6b9a695914005b9628
Question: In what country is Normandy located?

SCENARIO 1: WITH CONTEXT

[Context Text] (742 chars):
The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("...

[Full Prompt] (832 chars):
<s>[INST] Context:
The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norw...

[Token Positions]
  Total tokens in prompt: 206
  Context token indices: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 6

In [14]:
# Process all samples
from tqdm import tqdm
from pathlib import Path

def process_all_samples(
    samples,
    model,
    tokenizer,
    output_dir="./probe_out_mistral_context_comparison",
    mid_layer_index=16,
    max_samples=None,
    debug_first_n=0
):
    """
    Process all samples and save results.
    
    Args:
        samples: List of sample dicts
        model: XMistral model
        tokenizer: Tokenizer
        output_dir: Directory to save results
        mid_layer_index: Which layer to use for mid-layer features
        max_samples: Maximum number of samples to process (None = all)
        debug_first_n: Show debug output for first N samples (0 = no debug)
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    results_file = output_dir / "context_features.jsonl"
    
    n_success = 0
    n_fail = 0
    
    with results_file.open("w", encoding="utf-8") as f:
        for i, sample in enumerate(tqdm(samples, desc="Processing samples")):
            if max_samples is not None and i >= max_samples:
                break
            
            try:
                sample_id = sample.get("id", None)
                
                # Enable debug for first N samples
                show_debug = (debug_first_n > 0 and i < debug_first_n)
                if show_debug:
                    print(f"\n{'='*80}")
                    print(f"DEBUG: Sample {i+1}/{debug_first_n} - ID: {sample_id}")
                    print(f"{'='*80}")
                
                features = extract_context_features(
                    sample, model, tokenizer, mid_layer_index, debug=show_debug
                )
                
                # Prepare output row (flat structure like original)
                row = {
                    "id": sample_id,
                    "question": sample["question"],
                    "gold_answer": sample.get("gold_answer", ""),
                    "task_type": "open_qa",
                }
                
                # Add metrics at top level (matching original pattern)
                row["mid_context_metrics"] = features["mid_context_metrics"]
                row["last_context_metrics"] = features["last_context_metrics"]
                row["mid_no_context_metrics"] = features["mid_no_context_metrics"]
                row["last_no_context_metrics"] = features["last_no_context_metrics"]
                
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                n_success += 1
                
            except Exception as e:
                print(f"\nError processing sample {sample.get('id', 'unknown')}: {e}")
                f.write(json.dumps({"id": sample.get("id"), "error": str(e)}, ensure_ascii=False) + "\n")
                n_fail += 1
    
    print(f"\n{'='*60}")
    print(f"Processing complete!")
    print(f"  Success: {n_success}")
    print(f"  Failed: {n_fail}")
    print(f"  Output: {results_file}")
    print(f"{'='*60}")
    
    return results_file


## 📊 Process All Samples

**Debug Mode**: Set `debug_first_n=2` to see detailed output for first 2 samples.
This helps verify the pipeline is working correctly before running on the full dataset.


In [15]:
# Run processing on all samples (OUTPUT_DIR from config at top)
# Tip: Set debug_first_n=2 to see debug output for first 2 samples
output_file = process_all_samples(
    samples=samples_orig,
    model=model,
    tokenizer=tokenizer,
    output_dir=OUTPUT_DIR,
    mid_layer_index=16,
    max_samples=None,      # Set to a number for testing (e.g., 10)
    debug_first_n=0       # Set to 1-3 to see debug output for first N samples
)


Processing samples: 100%|██████████| 100/100 [00:33<00:00,  2.97it/s]


Processing complete!
  Success: 100
  Failed: 0
  Output: /app/xlong/scripts/data_preprocessing/runs/squad/context_features.jsonl


In [16]:
# Optional: Load and verify results
results = []
with open(output_file, "r", encoding="utf-8") as f:
    for line in f:
        results.append(json.loads(line))

print(f"Loaded {len(results)} results")
print("\nSample result structure:")
sample_row = results[0]
print(f"  ID: {sample_row['id']}")
print(f"  Question: {sample_row['question'][:50]}...")
print(f"  Has mid_context_metrics: {'mid_context_metrics' in sample_row}")
print(f"  Has last_context_metrics: {'last_context_metrics' in sample_row}")
print(f"  Has mid_no_context_metrics: {'mid_no_context_metrics' in sample_row}")
print(f"  Has last_no_context_metrics: {'last_no_context_metrics' in sample_row}")
print("\nFull sample (first result):")
print(json.dumps(results[0], indent=2))


Loaded 100 results

Sample result structure:
  ID: 56ddde6b9a695914005b9628
  Question: In what country is Normandy located?...
  Has mid_context_metrics: True
  Has last_context_metrics: True
  Has mid_no_context_metrics: True
  Has last_no_context_metrics: True

Full sample (first result):
{
  "id": "56ddde6b9a695914005b9628",
  "question": "In what country is Normandy located?",
  "gold_answer": "",
  "task_type": "open_qa",
  "mid_context_metrics": {
    "l2_first": 5.112834930419922,
    "l2_mean": 5.311837673187256,
    "l2_std": 0.33343783020973206,
    "hoyer_first": 0.32777345180511475,
    "hoyer_mean": 0.2975984215736389,
    "hoyer_std": 0.023307735100388527,
    "spec_entropy_first": 7.608030319213867,
    "spec_entropy_mean": 7.587812423706055,
    "spec_entropy_std": 0.013589396141469479,
    "excess_kurtosis_first": 27.20946502685547,
    "excess_kurtosis_mean": 14.504385948181152,
    "excess_kurtosis_std": 10.64127254486084
  },
  "last_context_metrics": {
    "l2_fir